In [1]:
import pandas as pd
import glob
import os

# Set the folder path where your CSVs are
folder_path = "Playlists"

# Use glob to get all CSV files in the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and combine them into one DataFrame
df_list = [pd.read_csv(file) for file in csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

combined_df

,#,Song,Artist,Popularity,BPM,Genres,Parent Genres,Album,Album Date,Time,...,Live,Loud (Db),Key,Time Signature,Added At,Spotify Track Id,Album Label,Camelot,ISRC,Time
0,1,Sweet Dreams (Are Made of This) - 2005 Remaster,"Eurythmics,Annie Lennox,Dave Stewart",85,125,"new wave, synthpop,","Pop, Rock, Electronic",Sweet Dreams (Are Made Of This),1983-01-04,03:36,...,10,-7,C minor,4,2021-08-13,1TfqLAPs4K3s2rJMoCokcS,RCA Records Label,5A,GBARL0300589,NaN
1,2,Smalltown Boy,Bronski Beat,3,135,"synthpop, hi-nrg","Electronic, Pop",The Age Of Consent,1984-00-00,05:03,...,20,-11,A#/B♭,4,2021-08-13,0FrCX7P2C2hcRTcuhjEvK4,Rhino/London-Sire,6B,GBAAP0200005,NaN
2,3,I'm Still Standing,Elton John,6,177,NaN,NaN,Rocket Man (Deluxe Edition),2007-03-26,03:01,...,30,-6,A#/B♭ minor,4,2021-08-13,0lzpfrTARexLFXEACKSXTh,Universal Music Group,3A,GBALX8300190,NaN
3,4,Funky Town,Lipps Inc.,6,122,disco,Pop,Paradas Continuas,2009-10-20,03:59,...,10,-8,C,4,2021-08-13,7723JnKU2R15Iv4T7OJrly,CONSECUENCIAS DISCOGRAFICAS,8B,MX1900900138,NaN
4,5,I'm So Excited,The Pointer Sisters,65,92,disco,Pop,Best Of,1989-10-21,03:49,...,10,-6,G#/A♭,4,2021-08-13,2u8MGAiS2hBVE7GZzTZLQI,Ariola,4B,USRC18203285,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8681,403,Super Freak,Rick James,75,132,"funk, motown, disco","Pop, R&B",Street Songs (Deluxe Edition),1981-04-07,03:25,...,0,-8,A minor,4,2025-03-17,2dCmGcEOQrMQhMMS8Vj7Ca,Motown,8A,USMO18100048,NaN
8682,404,Time After Time,Cyndi Lauper,72,130,NaN,NaN,She's So Unusual: A 30th Anniversary Celebrati...,2014-03-28,04:03,...,0,-9,C,4,2025-03-17,1Jj6MF0xDOMA3Ut2Z368Bx,Epic/Legacy,8B,USSM18300650,NaN
8683,405,Rock Me Amadeus,Falco,68,177,neue deutsche welle,"Pop, Rock",Falco 3,1985-02-19,03:22,...,0,-9,E minor,4,2025-03-17,0DfG1ltJnZyq4Tx3ZLL7ZU,Ariola,9A,ATB158500018,NaN
8684,406,Wicked Game,Chris Isaak,2,113,NaN,NaN,Best of Chris Isaak (Remastered),2006-00-00,04:46,...,0,-8,A,4,2025-03-17,390AWnOn2rfe9FzQjYmxIH,Mailboat Records,11B,USRE10601455,NaN


In [ ]:
# Count duplicates based on title and artist
duplicate_rows = combined_df.duplicated(subset=["Song", "Artist"]).sum()

print(f"🔎 Duplicate rows based on title and artist: {duplicate_rows}")

In [ ]:
clean_df = combined_df.drop_duplicates(subset=["Song", "Artist"], keep="first")

clean_df = clean_df.reset_index(drop=True)

clean_df

In [ ]:
# Remove rows where the title contains 'remix' (case-insensitive)
filtered_df = clean_df[~clean_df['Song'].str.contains("remix", case=False, na=False)]

filtered_df = filtered_df.reset_index(drop=True)

filtered_df

In [ ]:
columns_to_drop = ["Added At", "Spotify Track Id", "Album Label", "#", "ISRC", "Album Date", "Time ", "Album"]

# Drop them if they exist in the DataFrame
cleaner_df = filtered_df.drop(columns=[col for col in columns_to_drop if col in filtered_df.columns])

cleaner_df.columns

In [ ]:
# List of artists to exclude from splitting by comma
exceptions = ["Tyler, The Creator", "Earth, Wind & Fire"]

def clean_artist(artist_name):
    if artist_name in exceptions:
        # Return artist name as-is (no split)
        return artist_name.strip()
    else:
        # Otherwise, split on first comma and take only first part as primary artist
        # (or you can adjust logic if you want a list of artists)
        return artist_name.split(",")[0].strip()

# Apply to your DataFrame
cleaner_df["Artist_Clean"] = cleaner_df["Artist"].apply(clean_artist)
cleaner_df["Song_Clean"] = cleaner_df["Song"].str.split(" - ").str[0]


cleaner_df

In [ ]:
tyler_songs = cleaner_df[cleaner_df["Artist_Clean"] == "Tyler, The Creator"]
print(tyler_songs)

In [ ]:
cleaner_df.to_csv("initial_songs_master.csv", index=False)

## BREAK: Scrape genres

In [ ]:
# Load the CSV
genre_df = pd.read_csv("songs_with_genre_tags_full.csv")

# Get value counts for the 'Tag Source' column
tag_source_counts = genre_df["Tag Source"].value_counts()

# Display the result
print(tag_source_counts)


In [ ]:
none_tags_df = genre_df[genre_df["Tag Source"] == "none"]

# Count how many times each artist appears in the "none" category
artist_none_counts = none_tags_df["Artist"].value_counts()

# Display the top artists with missing tags
print(artist_none_counts.head(20))

In [ ]:
artist_tags_df = genre_df[genre_df["Tag Source"] == "artist"]

# Count how many times each artist appears in the "none" category
artist_artist_counts = artist_tags_df["Artist"].value_counts()

# Display the top artists with missing tags
print(artist_artist_counts.head(20))

In [ ]:
track_tags_df = genre_df[genre_df["Tag Source"] == "track"]

# Count how many times each artist appears in the "none" category
artist_track_counts = track_tags_df["Artist"].value_counts()

# Display the top artists with missing tags
print(artist_track_counts.head(20))

In [ ]:
# Keep only rows where Tag Source is 'track'
df_track_only = genre_df[genre_df["Tag Source"] == "track"]

# Optionally reset the index
df_track_only = df_track_only.reset_index(drop=True)

df_track_only

In [ ]:
# Save to a new CSV
df_track_only.to_csv("songs_genre_clean.csv", index=False)

## BREAK: Scrape lyrics

In [ ]:
df = pd.read_csv('songs_genre_lyrics.csv')

# Check how many rows have empty or missing lyrics
empty_lyrics = df[df['lyrics'].isna() | (df['lyrics'].str.strip() == '')]
print(f"Number of rows with empty lyrics: {len(empty_lyrics)}")

# Optionally preview them
print(empty_lyrics.head())


Number of rows with empty lyrics: 671
                                               Song  \
0   Sweet Dreams (Are Made of This) - 2005 Remaster   
15              Sisters Are Doin' It for Themselves   
21         Flashdance...What a Feeling - Radio Edit   
22         Got My Mind Set On You - Remastered 2004   
29        Let's Dance - Single Version [Remastered]   

                                               Artist  \
0                Eurythmics,Annie Lennox,Dave Stewart   
15  Eurythmics,Annie Lennox,Dave Stewart,Aretha Fr...   
21                                         Irene Cara   
22                                    George Harrison   
29                                        David Bowie   

                                         Genre Tags Tag Source lyrics  
0   80s, pop, new wave, female vocalists, synth pop      track    NaN  
15       80s, pop, new wave, rock, female vocalists      track    NaN  
21    80s, pop, Soundtrack, dance, female vocalists      track    NaN  


In [4]:

# Remove rows with empty or missing lyrics
df_cleaned = df[~(df['lyrics'].isna() | (df['lyrics'].str.strip() == ''))]

df_cleaned

,Song,Artist,Genre Tags,Tag Source,lyrics
1,Smalltown Boy,Bronski Beat,"80s, new wave, synthpop, pop, synth pop",track,To your soul\nTo your soul\nCry\nCry\nCry\n\nY...
2,I'm Still Standing,Elton John,"pop, 80s, elton john, rock, classic rock",track,You could never know what it's like\nYour bloo...
3,Funky Town,Lipps Inc.,"Disco, 80s, pop, 70s, dance",track,Gotta make a move to a town that's right for m...
4,I'm So Excited,The Pointer Sisters,"80s, Disco, pop, dance, soul",track,Tonight's the night we're gonna make it happen...
5,Cheri Cheri Lady,Modern Talking,"80s, Disco, pop, Modern Talking, dance",track,"Oh, I cannot explain\nEvery time, it's the sam..."
...,...,...,...,...,...
5460,Rock with You - Single Version,Michael Jackson,"pop, michael jackson, 80s, Disco, dance",track,"Girl, close your eyes\nLet that rhythm get int..."
5462,You Sexy Thing,Hot Chocolate,"Disco, 70s, funk, soul, pop",track,I believe in miracles\nWhere're you from?\nYou...
5464,Get It On,T. Rex,"glam rock, 70s, classic rock, rock, glam",track,"Well, you're dirty and sweet\nClad in black, d..."
5466,Love Really Hurts Without You,Billy Ocean,"80s, pop, soul, 70s, Disco",track,You run around town like a fool and you think ...


In [8]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Make language detection deterministic (same result every time)
DetectorFactory.seed = 0

# Function to detect language safely
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return 'error'

# Apply to the lyrics column (this may take some time with 2000+ songs)
df_cleaned['language'] = df_cleaned['lyrics'].apply(detect_language)

# Check how many are not English
non_english_df = df_cleaned[df_cleaned['language'] != 'en']

# Show some examples
print(non_english_df[['Song', 'Artist', 'language', 'lyrics']])

                             Song                       Artist language  \
57                 99 Luftballons                         Nena       de   
227               Rock Me Amadeus                        Falco       de   
304                      Bamboléo                  Gipsy Kings       es   
404                       Flicker              Porter Robinson       ja   
417                            OK                       Madeon       sk   
...                           ...                          ...      ...   
5173                         Tusa          KAROL G,Nicki Minaj       es   
5185                   Supalonely          BENEE,Gus Dapperton       tr   
5193                      Dilemma          Nelly,Kelly Rowland       de   
5253             Somethin' Stupid  Frank Sinatra,Nancy Sinatra       tr   
5434  Never Had A Dream Come True                Stevie Wonder       pt   

                                                 lyrics  
57    Hast du etwas Zeit für mich?\nDann 

/var/folders/nf/jqpg66kx52z7vrjhxgwb3h480000gn/T/ipykernel_70991/1616219158.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['language'] = df_cleaned['lyrics'].apply(detect_language)


In [11]:
# Assuming the column used to match is 'title'
df_final = df_cleaned[~df_cleaned['Song'].isin(non_english_df['Song'])]

df_final = df_final.reset_index(drop=True)

df_final

,Song,Artist,Genre Tags,Tag Source,lyrics,language
0,Smalltown Boy,Bronski Beat,"80s, new wave, synthpop, pop, synth pop",track,To your soul\nTo your soul\nCry\nCry\nCry\n\nY...,en
1,I'm Still Standing,Elton John,"pop, 80s, elton john, rock, classic rock",track,You could never know what it's like\nYour bloo...,en
2,Funky Town,Lipps Inc.,"Disco, 80s, pop, 70s, dance",track,Gotta make a move to a town that's right for m...,en
3,I'm So Excited,The Pointer Sisters,"80s, Disco, pop, dance, soul",track,Tonight's the night we're gonna make it happen...,en
4,Cheri Cheri Lady,Modern Talking,"80s, Disco, pop, Modern Talking, dance",track,"Oh, I cannot explain\nEvery time, it's the sam...",en
...,...,...,...,...,...,...
4484,Rock with You - Single Version,Michael Jackson,"pop, michael jackson, 80s, Disco, dance",track,"Girl, close your eyes\nLet that rhythm get int...",en
4485,You Sexy Thing,Hot Chocolate,"Disco, 70s, funk, soul, pop",track,I believe in miracles\nWhere're you from?\nYou...,en
4486,Get It On,T. Rex,"glam rock, 70s, classic rock, rock, glam",track,"Well, you're dirty and sweet\nClad in black, d...",en
4487,Love Really Hurts Without You,Billy Ocean,"80s, pop, soul, 70s, Disco",track,You run around town like a fool and you think ...,en


In [ ]:
master_df = pd.read_csv('initial_songs_master.csv')

# Merge Genre Tags and lyrics from df_final
# We'll assume the columns to join on are 'artist' and 'title'
columns_to_add = ['Artist', 'Song', 'Genre Tags', 'lyrics']  # adjust names if needed

# Ensure df_final has only the necessary columns
df_to_merge = df_final[columns_to_add]

# Perform the merge
merged_df = master_df.merge(df_to_merge, on=['Artist', 'Song'], how='left')

In [16]:
# Remove rows where lyrics are NaN or just whitespace
final_df = merged_df[~(merged_df['lyrics'].isna() | (merged_df['lyrics'].str.strip() == ''))]

# Reset index for cleanliness
final_df = final_df.reset_index(drop=True)

In [18]:
final_df = final_df.drop(columns=['Genres', 'Parent Genres', 'Time Signature'])

In [19]:
final_df

,Song,Artist,Popularity,BPM,Time,Dance,Energy,Acoustic,Instrumental,Happy,Speech,Live,Loud (Db),Key,Camelot,Artist_Clean,Song_Clean,Genre Tags,lyrics
0,Smalltown Boy,Bronski Beat,3,135,05:03,68,56,53,7,93,0,20,-11,A#/B♭,6B,Bronski Beat,Smalltown Boy,"80s, new wave, synthpop, pop, synth pop",To your soul\nTo your soul\nCry\nCry\nCry\n\nY...
1,I'm Still Standing,Elton John,6,177,03:01,49,93,46,0,79,10,30,-6,A#/B♭ minor,3A,Elton John,I'm Still Standing,"pop, 80s, elton john, rock, classic rock",You could never know what it's like\nYour bloo...
2,Funky Town,Lipps Inc.,6,122,03:59,91,63,0,62,34,0,10,-8,C,8B,Lipps Inc.,Funky Town,"Disco, 80s, pop, 70s, dance",Gotta make a move to a town that's right for m...
3,I'm So Excited,The Pointer Sisters,65,92,03:49,69,86,10,0,69,0,10,-6,G#/A♭,4B,The Pointer Sisters,I'm So Excited,"80s, Disco, pop, dance, soul",Tonight's the night we're gonna make it happen...
4,Cheri Cheri Lady,Modern Talking,82,114,03:46,68,62,46,1,85,0,30,-14,A#/B♭,6B,Modern Talking,Cheri Cheri Lady,"80s, Disco, pop, Modern Talking, dance","Oh, I cannot explain\nEvery time, it's the sam..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4484,Rock with You - Single Version,Michael Jackson,82,114,03:40,81,54,18,0,85,0,10,-13,C♯/D♭,3B,Michael Jackson,Rock with You,"pop, michael jackson, 80s, Disco, dance","Girl, close your eyes\nLet that rhythm get int..."
4485,You Sexy Thing,Hot Chocolate,75,106,04:04,79,73,52,0,96,0,10,-5,F,7B,Hot Chocolate,You Sexy Thing,"Disco, 70s, funk, soul, pop",I believe in miracles\nWhere're you from?\nYou...
4486,Get It On,T. Rex,72,127,04:22,73,88,18,86,91,0,60,-7,B minor,10A,T. Rex,Get It On,"glam rock, 70s, classic rock, rock, glam","Well, you're dirty and sweet\nClad in black, d..."
4487,Love Really Hurts Without You,Billy Ocean,1,141,02:58,52,91,0,0,96,0,10,-4,F,7B,Billy Ocean,Love Really Hurts Without You,"80s, pop, soul, 70s, Disco",You run around town like a fool and you think ...


In [ ]:
final_df.to_csv('dataset.csv', index=False)